In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import torch
import numpy as np
import matplotlib.pyplot as plt
import JHTDB_data_loading
import h5py
import os

# Data Format:
h5py.File(f'data/turbulence_output/channel_t={i}.h5', 'r')[f'Velocity_{i:04}'].shape==(77, 26, 103, 3) \
for 1<=i<=4000

In [ ]:
with h5py.File(f'data/turbulence_output/channel_t=1.h5', 'r') as f:
    print(f.keys())

In [ ]:
with h5py.File(f'data/turbulence_output/channel_t=1.h5', 'r') as f:
    print(f['Velocity_0001'].shape)

In [ ]:
with h5py.File(f'data/turbulence_output/channel_t=2.h5', 'r') as f:
    print(f['Velocity_0002'].shape)

## Make Fake Dataset:

In [ ]:
os.system('mkdir -p data/fake_turbulence_output')

for i in range(4009):
    i+=1
    new_file = h5py.File(f'data/fake_turbulence_output/channel_t={i}.h5', 'w')
    new_file[f'Velocity_{i:04}'] = np.random.rand(5, 3, 4, 3)
    new_file.close()

# Wrap and Load Datasets:

In [ ]:
import JHTDB_data_loading
class XYDatasetWrapper:
    def __getitem__(self, index):
        x, y = super().__getitem__(index)
        return torch.cat([x[...,None], y],axis=-1) # recombine!

class JHTDB_ChannelXY(XYDatasetWrapper, JHTDB_data_loading.JHTDB_Channel): pass
class JHTDB_ChannelBaselineXY(XYDatasetWrapper, JHTDB_data_loading.JHTDB_ChannelBaseline): pass

In [ ]:
time_stride = 2
DA_dataset_no_stride = JHTDB_ChannelXY('data/fake_turbulence_output', time_chunking=10)
DA_dataset = JHTDB_ChannelXY('data/fake_turbulence_output', time_stride=time_stride)
baseline_dataset = JHTDB_ChannelBaselineXY('data/fake_turbulence_output', time_stride=time_stride)

# Visualization Code:

In [ ]:
import numpy as np
def make_4d_sim_fig(sim_data, vel_comp_idx:int=0, prefix='', time_stride=time_stride, 
                    img_values_range: tuple|None=None, num_z=3, 
                    vel_component_name = ['X','Y','Z'], show=True):
    from grid_figures import GridFigure
    fig = GridFigure(f'{prefix}3d Channel Flow: {vel_component_name[vel_comp_idx]} Velocity')
    sim_data = np.asarray(sim_data)
    time_samples = list(range(sim_data.shape[-1])) if sim_data.shape[-1]<=10 else None
    for z in np.linspace(0, sim_data.shape[-2]-1, num=num_z, dtype=int):
        fig.add_3d_row(sim_data[vel_comp_idx,:,:,z], f'{z=}', x_title_func=lambda t: f't={t*time_stride}',
                       img_getter=lambda array_3d, t: array_3d[:,:,t].T, time_samples=time_samples)
    if img_values_range:
        fig._img_values_range = img_values_range
    if show: fig.show()
    return fig

def visualize_datum(dataset, index, **kwargs):
    figure = make_4d_sim_fig(dataset[index], time_stride=dataset.time_stride,
                             img_values_range=(0, 1), **kwargs)
    return figure

In [ ]:
visualize_datum(DA_dataset, 0)

# Visualize DA with Stride VS No Stride and Assert Data Augmentation Sliding Window Property

In [ ]:
DA_dataset[0].shape # = [channel, x, y, z, time]

In [ ]:
assert time_stride == 2
for i in range(10):
    visualize_datum(DA_dataset, i, prefix='with stride')
    visualize_datum(DA_dataset_no_stride, i, prefix='no stride')
    
    # check that data augmentation is correct moving the time dimension one step forward
    assert torch.allclose(DA_dataset_no_stride[i][...,1:], DA_dataset_no_stride[i+1][...,:-1]) # this tests that data aug exists (1 step at a time)
    assert torch.allclose(DA_dataset[i][...,1:], DA_dataset[i+2][...,:-1]) # this tests that data aug exists (1 step at a time)
    assert torch.allclose(DA_dataset_no_stride[i][...,::2], DA_dataset[i]) # this tests that stride is correct
    assert torch.allclose(DA_dataset_no_stride[i][...,1::2], DA_dataset[i+1]) # this tests that next timestep of strided dataset gets the values inbetween

# Visualize Baseline (Mutually Exclusive) VS Data Augmentation:

In [ ]:
for i in range(10):
    print(f'{i=}')
    visualize_datum(DA_dataset, i, prefix='With Data Augmentation: ')
    visualize_datum(baseline_dataset, i, prefix='Mutually Exclusive: ')

# Check Superset Property

In [ ]:
# Verify DA_dataset supersets baseline_dataset
hash_fn = lambda x: hash(str(x)) # can't use sum because of floating point precision issues

superset_checksums = set()
for i in range(len(DA_dataset)):
    superset_checksums.add(hash_fn(DA_dataset[i]))
for i in range(len(baseline_dataset)):
    assert hash_fn(baseline_dataset[i]) in superset_checksums

In [ ]:
len(superset_checksums)

# Check Length Calculation:
NOTE: requires temporarily disabling length based error checking inside corresponding __getitem__ methods

In [ ]:
DA_dataset[len(DA_dataset)]

In [ ]:
baseline_dataset[len(baseline_dataset)]